# Sesión de Trabajo 4: CORRELACIONES SIMPLES ENTRE INPUTS - OUPUTS
**Asignatura:** Ciència de Dades i Intel·ligència Artificial Aplicades a la Construcció i Estructures  
**Institución:** ETSEIB - UPC  

---

## Objetivos de la sesión
Hasta ahora habéis explorado los datos con el objetivo principal de entender de qué datos disponemos/no disponemos en referencia a la temática de estudio. En base a esta exploración realizada, habéis adquirido criterios para limpiar la base de datos original y ahora disponéis de una base de datos apta para el análisis de la sostenibilidad del parque edificado catalán en base a correlaciones simples.

El objetivo de la sesión consiste en extraer toda la información posible contenida en nuestra base de datos sobre la sostenibilidad del parque edificado catalán. Para ello, vamos a analizar cómo influyen cada una de las variables de entrada a los outputs disponibles.

* **S4.T1.** Correlaciones entre 2 variables categóricas

* **S4.T2.** Correlaciones entre 1 variable continua y una variable categórica


---



### S4.T0. Importar **TU** base de datos **limpia**
En esta sesión vamos a trabajar con la base de datos que tu grupo limpió y guardó al final de la Sesión 3. Seguid estos pasos:

* **Un miembro del grupo** debe subir el archivo BBDD_ST3.csv a su Google Drive Personal.
* Haced clic derecho sobre el archivo > Compartir > Compartir.
* En "Acceso general", cambiad "Restringido" por "Cualquier persona con el enlace". Aseguraos de que a la derecha aparezca el rol de Lector.
* Haz clic en "Copiar enlace".

¿Cómo extraer el ID del enlace?

El enlace que has copiado tendrá este aspecto:
https://drive.google.com/file/d/16TJBJiLZOKiQBsi4mX14mJ1Nhu736Gpv/view?usp=sharing

El ID es la cadena de caracteres que hay entre /d/ y /view.

Ejemplo: Si vuestro enlace es https://drive.google.com/file/d/16TJBJiLZOKiQBsi4mX14mJ1Nhu736Gpv/view?usp=drive_link, vuestro ID es: 16TJBJiLZOKiQBsi4mX14mJ1Nhu736Gpv


⏳ *Tiempo de ejecución estimado: 1-2 minutos (dependiendo de la velocidad de conexión a Drive).*



In [ ]:
# Importar librerías
!pip install --upgrade -q gdown
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import gdown
import os

# ==========================================================
# 2. Configuración con tu ID de archivo (por grupo)
# Pegad aquí el ID que habéis extraído de vuestro enlace de Drive
file_id = 'PEGAR_AQUÍ_VUESTRO_ID'
# ==========================================================

url = f'https://drive.google.com/uc?id={file_id}'
output = 'BBDD_ST3.csv'

# 3. Limpieza de archivos corruptos previos
if os.path.exists(output):
    os.remove(output)
    print(f"Borrando versión antigua de {output} para evitar conflictos...")

# 4. Descarga robusta
print("Iniciando la descarga del dataset limpio vía Google Drive Personal...")
gdown.download(url, output, quiet=False)

print(f"Descarga completada: {os.path.getsize(output) / (1024*1024):.2f} MB")
print("Cargando datos en el sistema...")

# Cargamos el CSV
df = pd.read_csv(output, low_memory=False)
print("¡Base de datos lista para trabajar!")
display(df.head())

# Optimización
df["CODI_POSTAL"] = df["CODI_POSTAL"].astype('str')
df["VALOR AILLAMENTS CTE"] = df["VALOR AILLAMENTS CTE"].astype('str')
df["VALOR FINESTRES CTE"] = df["VALOR FINESTRES CTE"].astype('str')
float64_cols = df.select_dtypes(include='float64').columns.tolist()
df[float64_cols] = df[float64_cols].astype('float32')
int64_cols = df.select_dtypes(include='int64').columns.tolist()
df[int64_cols] = df[int64_cols].astype('int32')

print("Base de datos cargada y optimizada.")

---

### S4.T1. Visualizar correlaciones entre variables categóricas

* Explora las correlaciones simples entre cada una de las variables categóricas de entrada y el output categórico de vuestra elección:

`Qualificació de consum d'energia primaria no renovable` o `Qualificacio d'emissions de CO2`

* Analiza e interpreta las correlaciones obtenidas para obtener conocimiento sobre el estado energético del parque edificado catalán.

---


In [ ]:
# EJEMPLO DE ANÁLISIS: Normativa construcció vs Qualificació de consum
print("DISTRIBUCIÓN DE REGISTROS POR NORMATIVA:")
print(df["Normativa construcció"].value_counts(dropna=False))

# 1. Definir la variable de salida (Target)
target = "Qualificació de consum d'energia primaria no renovable"

# 2. Visualización comparativa de todas las normativas
aux = df.groupby('Normativa construcció')[target].value_counts(normalize=True).mul(100)
orden_norm = ["abans de 1979", "nbe-ct-79", "nre-at-87", "cte 2006", "cte 2013"]

# Usamos unstack y reindexamos para asegurar el orden cronológico en el gráfico
ax = aux.unstack().reindex(orden_norm).plot.bar(stacked=True, rot=0, figsize=(10, 5))
ax.legend(title="Calificación", bbox_to_anchor=(1.0, 1.0), loc='upper left')

In [ ]:
# EJEMPLO DE ANÁLISIS: PIS vs Qualificació de consum
print("ESTADO DE LA VARIABLE 'PIS':")
print(df["PIS"].value_counts(dropna=False))

# 1. Definir la variable de salida (Target)
target = "Qualificació de consum d'energia primaria no renovable"

# 2. Cálculo de la distribución por tipo de planta
aux = df.groupby("PIS")[target].value_counts(normalize=True).mul(100)

# 3. Visualización ordenada por altura del edificio
print("\nGENERANDO GRÁFICO POR ALTURA...")
orden_pis = ["SOT", "BX", "EN", "PR", "1", "2", "3", "4", "5-9", ">10", "AT", "SA"]
ax = aux.unstack().reindex(orden_pis).plot.bar(stacked=True, figsize=(10, 5))
ax.legend(title="Calificación", bbox_to_anchor=(1.0, 1.0), loc='upper left')

In [ ]:
# EJEMPLO DE ANÁLISIS: COMARCA vs Eficiencia
print("DISTRIBUCIÓN POR COMARCAS (Top 10):")
print(df["COMARCA"].value_counts(dropna=False).head(10))

target = "Qualificació de consum d'energia primaria no renovable"

aux = df.groupby("COMARCA")[target].value_counts(normalize=True).mul(100)

ax = aux.unstack().plot.bar(stacked=True, rot=90, figsize=(15, 6))
ax.legend(title="Calificación", bbox_to_anchor=(1.0, 1.0), loc='upper left')

>TAREA: Explora las correlaciones entre las demás variables categóricas (crea tantas celdas de código como necesites).


In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable categórica 1 y el output elegido

In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable categórica 2 y el output elegido

In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable categórica 3 y el output elegido

---

### S4.T2. Visualizar correlaciones entre 1 input continuo con 1 output categórico

* Explora las correlaciones simples entre cada una de las variables continuas de entrada y el output categórico de vuestra elección:

`Qualificació de consum d'energia primaria no renovable` o `Qualificacio d'emissions de CO2`

* Analiza e interpreta las correlaciones obtenidas para obtener conocimiento sobre el estado energético del parque edificado catalán.

---


In [ ]:
# 1. Definir la variable de salida (Target)
target = "Qualificació de consum d'energia primaria no renovable"

# 2. Configuración del gráfico de caja (Boxplot)
plt.figure(figsize=(10, 6))

# showfliers=False: oculta los valores atípicos para que el gráfico sea legible
# showmeans=True: añade un marcador para la media aritmética
sns.boxplot(
    x=target,
    y="METRES_CADASTRE",
    data=df,
    order=["A", "B", "C", "D", "E", "F", "G"],
    showfliers=False,
    showmeans=True,
    palette="RdYlGn_r"  # Paleta de color de verde (A) a rojo (G)
)

# 3. Personalización de etiquetas
plt.title("Distribución de Superficie (Catastro) según Calificación Energética")
plt.xlabel("Calificación Energética")
plt.ylabel("m²")
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

>TAREA: Explora las correlaciones entre las demás variables continuas (crea tantas celdas de código como necesites).


In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable continua 1 y el output elegido

In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable continua 2 y el output elegido

In [ ]:
#Insertar código aquí para la visualización de la correlación entre la variable continua 3 y el output elegido